# 03 Model Evaluation
Evaluating the detection algorithms with Sliding Window and ADT metric.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (parent / 'src').exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [4]:
import numpy as np
import pickle
from src.config import MODELS_DIR, SYNTHETIC_DATA_DIR, SAMPLING_RATE_HZ
from src.models.sliding_window import SlidingWindowDetector
from src.evaluation.metrics import compute_advance_detection_time

with open(MODELS_DIR / 'sktime_mlp_classifier.pkl', 'rb') as f:
    clf = pickle.load(f)

# Load 10 events for evaluation
events = np.load(SYNTHETIC_DATA_DIR / 'event_samples.npy')[:10]
timestamps = np.load(SYNTHETIC_DATA_DIR / 'event_timestamps.npy')[:10]

detector = SlidingWindowDetector(clf)

all_adts = []
valid_peaks = []
for i in range(len(events)):
    signal = events[i]
    peak = timestamps[i]
    detections, proba, centers = detector.detect(signal)
    res = compute_advance_detection_time(detections, [peak], SAMPLING_RATE_HZ)
    if res['per_event_adt_ms']:
        all_adts.extend(res['per_event_adt_ms'])
        valid_peaks.append(peak)

print(f'Evaluated on {len(valid_peaks)}/{len(events)} events.')
if len(all_adts) > 0:
    print(f'Average Advance Detection Time (ADT): {np.mean(all_adts):.2f} ms')

Evaluated on 2/10 events.
Average Advance Detection Time (ADT): 24.50 ms
